# Helmet non-compliance detection — v2 experiments
Dataset: Kaggle `andrewmvd/hard-hat-detection` (5,000 images, Pascal VOC).

What this notebook produces (all saved to Google Drive, so a disconnect does not lose work):
1. 2-class dataset (`helmet`, `head` = bare head) with a fixed 70/15/15 train/val/test split
2. Training of YOLOv5s, YOLOv8s and YOLO11s under identical settings
3. Test-set metrics (overall and per class), GPU and CPU speed
4. Alarm-level evaluation: confidence threshold chosen on **val**, reported on **test** (missed violations / false alarms)
5. Optional: alarm logic on your own video

**Runtime → Change runtime type → T4 GPU** before running. Total time ≈ 3–5 h on a T4 (cells resume if Colab disconnects: just re-run all).

In [ ]:
!pip -q install ultralytics kagglehub
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
OUT = Path('/content/drive/MyDrive/helmet_v2')   # everything is saved here
RUNS = OUT / 'runs'
OUT.mkdir(parents=True, exist_ok=True); RUNS.mkdir(exist_ok=True)
SEED = 42
IMGSZ = 416          # native resolution of the dataset images
CLASSES = ['helmet', 'head']
MODELS = {'YOLOv5s': 'yolov5su.pt', 'YOLOv8s': 'yolov8s.pt', 'YOLO11s': 'yolo11s.pt'}

## 1. Download dataset
If the download asks for credentials: Kaggle → Settings → *Create New Token*, then in Colab click the 🔑 (Secrets) icon and add `KAGGLE_USERNAME` and `KAGGLE_KEY`.

In [ ]:
import os, kagglehub
try:
    from google.colab import userdata
    for k in ('KAGGLE_USERNAME', 'KAGGLE_KEY'):
        try: os.environ[k] = userdata.get(k)
        except Exception: pass
except Exception:
    pass
SRC = Path(kagglehub.dataset_download('andrewmvd/hard-hat-detection'))
print(SRC, '| xml:', len(list(SRC.rglob('*.xml'))))

## 2. Convert VOC → YOLO (2 classes) and split 70/15/15

In [ ]:
import random, shutil, csv
import xml.etree.ElementTree as ET
from collections import Counter
from PIL import Image

DATA = Path('/content/hardhat_yolo')
IMG_EXT = ('.png', '.jpg', '.jpeg')

def voc_to_yolo(xml_path, img_path, classes):
    root = ET.parse(xml_path).getroot()
    size = root.find('size')
    w = float(size.find('width').text) if size is not None else 0
    h = float(size.find('height').text) if size is not None else 0
    if w <= 0 or h <= 0:
        with Image.open(img_path) as im: w, h = im.size
    lines, counts = [], Counter()
    for obj in root.iter('object'):
        name = obj.find('name').text.strip().lower()
        if name not in classes:
            continue                         # 'person' is dropped
        bb = obj.find('bndbox')
        x1, y1, x2, y2 = (float(bb.find(t).text) for t in ('xmin', 'ymin', 'xmax', 'ymax'))
        x1, x2 = max(0, min(x1, x2)), min(w, max(x1, x2))
        y1, y2 = max(0, min(y1, y2)), min(h, max(y1, y2))
        if x2 - x1 < 1 or y2 - y1 < 1:
            continue
        c = classes.index(name)
        lines.append(f"{c} {(x1+x2)/2/w:.6f} {(y1+y2)/2/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}")
        counts[name] += 1
    return lines, counts

xmls = sorted(SRC.rglob('*.xml'))
imgs = {p.stem: p for p in SRC.rglob('*') if p.suffix.lower() in IMG_EXT}
pairs = [(x, imgs[x.stem]) for x in xmls if x.stem in imgs]
print('image/annotation pairs:', len(pairs))

split_file = OUT / 'split.csv'            # frozen split, reused on every run
if split_file.exists():
    assign = {r['stem']: r['split'] for r in csv.DictReader(open(split_file))}
else:
    rng = random.Random(SEED); order = pairs[:]; rng.shuffle(order)
    n = len(order); n_tr, n_va = int(0.70 * n), int(0.15 * n)
    assign = {x.stem: ('train' if i < n_tr else 'val' if i < n_tr + n_va else 'test')
              for i, (x, _) in enumerate(order)}
    with open(split_file, 'w', newline='') as f:
        wr = csv.writer(f); wr.writerow(['stem', 'split']); wr.writerows(assign.items())

if DATA.exists(): shutil.rmtree(DATA)
stats = {s: Counter() for s in ('train', 'val', 'test')}
for x, im in pairs:
    s = assign[x.stem]
    (DATA / 'images' / s).mkdir(parents=True, exist_ok=True)
    (DATA / 'labels' / s).mkdir(parents=True, exist_ok=True)
    lines, cnt = voc_to_yolo(x, im, CLASSES)
    shutil.copy(im, DATA / 'images' / s / im.name)
    (DATA / 'labels' / s / f'{x.stem}.txt').write_text('\n'.join(lines))
    stats[s]['images'] += 1; stats[s].update(cnt)
    if cnt['head'] > 0: stats[s]['images_with_bare_head'] += 1

(DATA / 'data.yaml').write_text(
    f"path: {DATA}\ntrain: images/train\nval: images/val\ntest: images/test\n"
    f"names:\n  0: helmet\n  1: head\n")

import pandas as pd
ds = pd.DataFrame(stats).T.fillna(0).astype(int)
ds.loc['total'] = ds.sum()
ds.to_csv(OUT / 'table_dataset.csv'); ds

## 3. Train the three models (identical settings, early stopping)

In [ ]:
from ultralytics import YOLO

for name, w in MODELS.items():
    run = RUNS / name
    if (run / 'DONE').exists():
        print(name, 'already trained'); continue
    last = run / 'weights' / 'last.pt'
    if last.exists():
        print(name, 'resuming'); YOLO(str(last)).train(resume=True)
    else:
        YOLO(w).train(data=str(DATA / 'data.yaml'), epochs=100, patience=20, imgsz=IMGSZ,
                      batch=16, seed=SEED, deterministic=True, optimizer='auto',
                      project=str(RUNS), name=name, exist_ok=True, plots=True)
    (run / 'DONE').write_text('ok')

## 4. Test-set evaluation (overall + per class) and speed

In [ ]:
import time, glob, numpy as np, torch

test_imgs = sorted(glob.glob(str(DATA / 'images' / 'test' / '*')))
def fps(model, device, n=100):
    ims = test_imgs[:n]
    model.predict(ims[:5], imgsz=IMGSZ, device=device, verbose=False)   # warm-up
    t = time.perf_counter()
    for p in ims: model.predict(p, imgsz=IMGSZ, device=device, verbose=False)
    return n / (time.perf_counter() - t)

rows = []
for name in MODELS:
    m = YOLO(str(RUNS / name / 'weights' / 'best.pt'))
    r = m.val(data=str(DATA / 'data.yaml'), split='test', imgsz=IMGSZ, batch=16,
              project=str(RUNS), name=f'{name}_test', exist_ok=True, plots=True, verbose=False)
    row = {'model': name, 'P': r.box.mp, 'R': r.box.mr, 'mAP50': r.box.map50, 'mAP50-95': r.box.map}
    for j, c in enumerate(r.box.ap_class_index):
        p, rc, a50, a = r.box.class_result(j)
        cn = CLASSES[int(c)]
        row.update({f'{cn}_P': p, f'{cn}_R': rc, f'{cn}_AP50': a50, f'{cn}_AP50-95': a})
    row['params_M'] = sum(x.numel() for x in m.model.parameters()) / 1e6
    row['epochs_run'] = len(pd.read_csv(RUNS / name / 'results.csv'))
    row['FPS_GPU'] = fps(m, 0) if torch.cuda.is_available() else np.nan
    row['FPS_CPU'] = fps(m, 'cpu', n=50)
    rows.append(row)

test_tab = pd.DataFrame(rows).set_index('model').round(3)
test_tab.to_csv(OUT / 'table_test_metrics.csv'); test_tab

## 5. Alarm-level evaluation
An image is a **violation** if it contains ≥1 bare head. The alarm fires if the model detects a `head` with confidence ≥ *t*.
*t* is selected on the **validation** set (max F1), then applied unchanged to the **test** set → no leakage.

In [ ]:
def max_head_conf(model, split):
    lbl_dir = DATA / 'labels' / split
    gt, score = [], []
    for res in model.predict(str(DATA / 'images' / split), imgsz=IMGSZ, conf=0.05,
                             classes=[1], stream=True, verbose=False):
        stem = Path(res.path).stem
        txt = (lbl_dir / f'{stem}.txt').read_text().split('\n')
        gt.append(int(any(l.startswith('1 ') for l in txt)))
        score.append(float(res.boxes.conf.max()) if len(res.boxes) else 0.0)
    return np.array(gt), np.array(score)

def alarm_metrics(gt, score, t):
    pred = score >= t
    tp = int((pred & (gt == 1)).sum()); fn = int((~pred & (gt == 1)).sum())
    fp = int((pred & (gt == 0)).sum()); tn = int((~pred & (gt == 0)).sum())
    p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    return dict(t=t, TP=tp, FN_missed=fn, FP_false_alarm=fp, TN=tn, P=p, R=r, F1=f1,
                false_alarm_rate=fp / (fp + tn) if fp + tn else 0.0)

TS = np.round(np.arange(0.05, 0.96, 0.05), 2)
alarm_rows, curves = [], {}
for name in MODELS:
    m = YOLO(str(RUNS / name / 'weights' / 'best.pt'))
    gv, sv = max_head_conf(m, 'val')
    sweep = pd.DataFrame([alarm_metrics(gv, sv, t) for t in TS]); curves[name] = sweep
    sweep.to_csv(OUT / f'alarm_sweep_val_{name}.csv', index=False)
    t_best = float(sweep.loc[sweep.F1.idxmax(), 't'])
    gt_, st_ = max_head_conf(m, 'test')
    alarm_rows.append({'model': name, **alarm_metrics(gt_, st_, t_best)})

alarm_tab = pd.DataFrame(alarm_rows).set_index('model').round(3)
alarm_tab.to_csv(OUT / 'table_alarm_test.csv'); alarm_tab

## 6. Figures (300 dpi, saved to Drive)

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'serif', 'font.size': 10})

fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
for name, s in curves.items():
    ax[0].plot(s.t, s.F1, marker='o', ms=3, label=name)
    ax[1].plot(s.R, s.P, marker='o', ms=3, label=name)
ax[0].set(xlabel='Confidence threshold', ylabel='Alarm F1 (validation)', title='(a)')
ax[1].set(xlabel='Recall (violations caught)', ylabel='Precision (true alarms)', title='(b)')
for a in ax: a.grid(alpha=.3); a.legend(frameon=False)
fig.tight_layout(); fig.savefig(OUT / 'fig_alarm_threshold.png', dpi=300); plt.show()

fig, ax = plt.subplots(figsize=(6, 3.4))
cols = ['helmet_AP50', 'head_AP50', 'mAP50-95']
test_tab[cols].plot.bar(ax=ax, rot=0, width=.75)
ax.set(ylabel='Score (test set)', ylim=(0, 1)); ax.grid(axis='y', alpha=.3); ax.legend(frameon=False, ncol=3)
fig.tight_layout(); fig.savefig(OUT / 'fig_model_comparison.png', dpi=300); plt.show()

## 7. (Optional) Alarm logic on your own video
Upload a video (people with and without helmets). To avoid flicker, the alarm fires when a bare head is detected in ≥ `K` of the last `N` frames and stops when this is no longer true. Set `BEST` to the model chosen in section 5.

In [ ]:
from google.colab import files
from collections import deque
import cv2

BEST = alarm_tab.F1.idxmax(); T = float(alarm_tab.loc[BEST, 't']); K, N = 5, 10
up = files.upload(); vid = next(iter(up))
m = YOLO(str(RUNS / BEST / 'weights' / 'best.pt'))
cap = cv2.VideoCapture(vid); fps_v = cap.get(cv2.CAP_PROP_FPS) or 25
W, H = int(cap.get(3)), int(cap.get(4))
out = cv2.VideoWriter(str(OUT / f'alarm_{Path(vid).stem}.mp4'), cv2.VideoWriter_fourcc(*'mp4v'), fps_v, (W, H))
win, alarm, events, f, log = deque(maxlen=N), False, [], 0, []
while True:
    ok, frame = cap.read()
    if not ok: break
    r = m.predict(frame, imgsz=IMGSZ, conf=T, verbose=False)[0]
    win.append(int((r.boxes.cls == 1).any().item()) if len(r.boxes) else 0)
    now = sum(win) >= K
    if now and not alarm: events.append({'start_s': round(f / fps_v, 2)})
    if not now and alarm: events[-1]['end_s'] = round(f / fps_v, 2)
    alarm = now
    img = r.plot()
    if alarm: cv2.putText(img, 'ALARM: HELMET MISSING', (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
    out.write(img); f += 1
cap.release(); out.release()
ev = pd.DataFrame(events); ev.to_csv(OUT / f'alarm_events_{Path(vid).stem}.csv', index=False)
print(f'{f} frames, {len(events)} alarm events'); ev

## 8. Summary — send me these files from `MyDrive/helmet_v2`

In [ ]:
print(ds, '\n'); print(test_tab, '\n'); print(alarm_tab)
!cd "{OUT}" && zip -qr ../helmet_v2_results.zip . -x "runs/*/weights/*"
print('Zip ready: MyDrive/helmet_v2_results.zip')